# Birth–death model of one EcoBici station

This notebook works in three stages:

1. **Validate**: check that the assumptions behind a birth–death chain are reasonable for your station.
2. **Compute**: build the chain and compute what it tells you (probability of empty/full, hitting times, best starting stock).
3. **Evaluate**: test the fitted model on days it has not seen.

All functions live in `ecobici_bd.py` (keep it next to this notebook).

**Running it.** With `DATA_PATH = None` the notebook generates synthetic trips whose true rates are known, so you can see how each check behaves when the model is right. To use your data, set `DATA_PATH` and `STATION` in the next cell. Cells marked *synthetic only* compare against the truth and are skipped automatically on real data.

In [ ]:
import glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import ecobici_bd as bd

DATA_PATH = "df_last_two_months.csv"  # e.g. "data/ecobici_2024_*.csv", a list of paths, or None (synthetic demo)
STATION = "160"              # id as it appears in Ciclo_Estacion_Retiro / Ciclo_EstacionArribo
CAPACITY = 23                    # official dock count if you have it (GBFS station_information); None = estimate
HOLIDAYS = []                    # treated as weekend days, e.g. ["2024-02-05", "2024-03-18"]
BURN_IN_DAYS, BURN_OUT_DAYS = 3, 2   # edges where the reconstruction is unreliable
GRID_MIN = 15                    # time grid for exposures; bin widths must be multiples of it
TEST_FRAC = 0.2                  # last 20% of days held out for Stage 3
SYNTH_DAY_SD = 0.0               # synthetic only: >0 adds day-to-day rate variation (try 0.3)

plt.rcParams.update({"figure.figsize": (10, 3.6), "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2})
C1, C2, C3, C4 = bd.COLORS

## Stage 0 — Data and occupancy reconstruction

The birth–death chain is about $N(t)$, the number of docked bikes, but the data only lists trips. Everything below needs $N(t)$, so we rebuild it first.

**Births and deaths.** For station $i$, an arrival (birth) is a trip with `Ciclo_EstacionArribo = i` at time `Fecha/Hora_Arribo`. A departure (death) is a trip with `Ciclo_Estacion_Retiro = i` at time `Fecha/Hora_Retiro`.

**Stays.** Sort each bike's trips by time. Between trip $k$ and trip $k+1$ the bike is docked somewhere:

- if trip $k$ ends at station $s$ and trip $k+1$ starts at $s$, the bike was at $s$ during $[t^{\text{in}}_k,\ t^{\text{out}}_{k+1})$;
- if trip $k+1$ starts at a different station, a truck moved it (*rebalancing*) at an unknown time inside that window.

Occupancy is then a count of covering intervals:

$$N_i(t)=\sum_b \mathbf 1\{\text{bike } b \text{ has a stay at } i \text{ covering } t\}.$$

**Truck times.** For rebalancing moves at the station we assume the *fewest truck visits* that explain the data. Each move is only known to lie inside its window $[s_j, e_j]$. Sort the windows by $e_j$. The first window's end opens a visit, and every window with $s_j \le e_j$ joins it. All members contain that point, so their common intersection $[\max s_j,\ e_j]$ is non-empty, and the visit time is placed at its midpoint (greedy interval stabbing, which is optimal for this problem).

**Capacity.** If you don't have the official dock count, $\hat C$ is the most common daily maximum of $N(t)$: a busy station hits its true capacity on many days, while reconstruction errors push the maximum above it only occasionally.

**What can go wrong.** A truck that removes a bike and later brings the same bike back, with no ride in between, is invisible: it looks like one long stay. Bikes that never move (broken) are never seen. The checks below include a direct measure of reconstruction quality: events the model says are impossible (arrivals at a full station, departures from an empty one).

In [ ]:
if DATA_PATH is None:
    raw, TRUE_OCC = bd.simulate_trips(n_days=120, day_sd=SYNTH_DAY_SD, seed=1)
    STATION, SYNTH = "100", True
else:
    raw = sorted(glob.glob(DATA_PATH)) if isinstance(DATA_PATH, str) else DATA_PATH
    TRUE_OCC, SYNTH = None, False
    
df, rep = bd.load_trips(raw)

stays, moves, srep = bd.build_stays(df, STATION)
stays, unresolved = bd.enforce_capacity(stays, STATION, df.loc[df.to == STATION, "t_in"], CAPACITY)
print("arrivals still meeting a full station:", unresolved)
print({k: srep[k] for k in ("stays", "rebalancing_moves", "share_rebalanced", "overlapping_dropped")})

START = srep["data_start"].normalize() + pd.Timedelta(days=BURN_IN_DAYS)
END = srep["data_end"].normalize() - pd.Timedelta(days=BURN_OUT_DAYS)
occ_raw = bd.occupancy(stays, STATION)
C_est, daily_max = bd.estimate_capacity(occ_raw, START, END)
C = CAPACITY or C_est
print(f"capacity used C = {C}  (estimate {C_est}); daily maxima:", daily_max.to_dict())

occ = bd.clip_occupancy(occ_raw, C)
seg = bd.build_segments(occ, START, END, GRID_MIN, HOLIDAYS)
arr, dep = bd.station_events(df, STATION, occ, START, END, HOLIDAYS)
train, test = bd.chronological_split(seg.date, TEST_FRAC)
print(f"{len(arr)} arrivals, {len(dep)} departures; train {len(train)} days, test {len(test)} days")
print("impossible events (reconstruction errors): arrivals at N=C:",
      int((arr.N_before >= C).sum()), " departures at N=0:", int((dep.N_before <= 0).sum()))



In [ ]:
arr_raw = bd.annotate_events(df.loc[(df.to == STATION) & (df.t_in >= START) & (df.t_in < END), "t_in"], occ_raw)
seg_raw = bd.build_segments(occ_raw, START, END, GRID_MIN, HOLIDAYS)
day = (seg_raw.minute >= 6 * 60) & (seg_raw.minute < 23 * 60)          # service hours only
ev_day = (arr_raw.minute >= 6 * 60) & (arr_raw.minute < 23 * 60)
tab = pd.DataFrame({"hours": seg_raw[day].groupby("N").dur_h.sum(),
                    "arrivals": arr_raw[ev_day].groupby("N_before").size()}).fillna(0)
tab["arrivals_per_hour"] = tab.arrivals / tab.hours
print(tab.loc[C - 4: C + 6].round(2))

In [ ]:
# Occupancy over one week (and the truth, synthetic only)
w0 = train[7]; w1 = w0 + pd.Timedelta(days=7)
o = occ[(occ.t >= w0) & (occ.t < w1)]
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.step(o.t, o.N, where="post", color=C1, label="reconstructed")
if SYNTH:
    tr = TRUE_OCC[(TRUE_OCC.t >= w0) & (TRUE_OCC.t < w1)]
    ax.step(tr.t, tr.N, where="post", color=C2, lw=1.2, alpha=0.8, label="true")
ax.axhline(C, color="0.5", lw=1, ls="--"); ax.text(o.t.iloc[0], C + 0.3, "capacity", color="0.4", fontsize=9)
ax.set_ylabel("bikes docked"); ax.set_title(f"Station {STATION}: occupancy, one week"); ax.legend(loc="upper right")
plt.show()

if SYNTH:
    ts = pd.date_range(START, END, freq="7min")
    rec, tru = bd.n_at(occ, ts), np.clip(bd.n_at(TRUE_OCC, ts), 0, C)
    by_h = pd.DataFrame({"h": ts.hour, "err": rec - tru}).groupby("h").err.mean()
    print(f"[synthetic] exact match {np.mean(rec == tru):.1%}, mean abs error {np.abs(rec - tru).mean():.2f} bikes")
    print("[synthetic] mean error by hour (+ = overcount):", by_h.round(2).to_dict())

In [ ]:
st = stays[stays.station == STATION]
ev = pd.concat([pd.DataFrame({"t": st.start, "d": 1,  "kind": st.kind}),
                pd.DataFrame({"t": st.end,   "d": -1, "kind": st.kind})])
jumps = ev.groupby("t").agg(dN=("d", "sum"), kinds=("kind", lambda k: k.value_counts().to_dict()))
big = jumps[jumps.dN.abs() >= 5]
print(len(big), "instants where N changes by 5 or more")
print(big.sort_index().head(50))

# raw (unclipped) count, to see how far above 39 it really goes
print(occ_raw.N.describe())

## Stage 1 — Validate the assumptions

### 1.1 The model and what it assumes

The state is $N(t)\in\{0,\dots,C\}$. The chain is a continuous-time Markov chain with generator

$$q_{n,n+1}=\lambda(t)\,\mathbf 1\{n<C\},\qquad q_{n,n-1}=\mu(t)\,\mathbf 1\{n>0\},\qquad q_{nn}=-\textstyle\sum_{m\neq n}q_{nm}.$$

Each assumption is something the data can contradict:

| | Assumption | What breaks it | Check |
|---|---|---|---|
| A1 | Return and pickup **demand** are Poisson processes | groups, bursts | 1.5, 1.6, 3.1 |
| A2 | Rates are constant inside each time bin | rate changes within the bin | 1.3, 1.5 |
| A3 | Rates don't depend on $n$ (except at $0$ and $C$) | app avoidance, broken last bike | 1.7 |
| A4 | One event moves one bike | friends riding together | 1.6 |
| A5 | All days of the same type share the same rates | weather, events, holidays | 1.4 |

Why Poisson? A Markov chain needs memoryless waiting times. The only memoryless continuous distribution is the exponential, and exponential gaps define a Poisson process. A good reason to expect it here: many independent riders, each rarely using this station, add up to an approximately Poisson stream (Palm–Khintchine theorem).

### 1.2 The estimator: censoring-corrected Poisson MLE

Testing assumptions requires the model fitted under them, so the estimator comes first.

Demand for pickups is Poisson with rate $\mu$, but a pickup can only happen when $N(t^-)>0$. Observed departures form a counting process with intensity $\mu\,Y(t)$, where $Y(t)=\mathbf 1\{N(t^-)>0\}$. The likelihood of a counting process with intensity $\mu Y(t)$ observed on $[0,T]$, with $k$ events, is

$$L(\mu)=\prod_{j=1}^{k}\mu\,Y(t_j)\;\exp\!\Big(-\mu\int_0^T Y(t)\,dt\Big),$$

so $\ell(\mu)=k\log\mu-\mu E$, with exposure $E=\int_0^T Y(t)\,dt$. Setting $\ell'=0$ gives

$$\hat\mu=\frac{k}{E}=\frac{\#\text{departures}}{\text{time with at least one bike}},\qquad \operatorname{SE}(\hat\mu)\approx\frac{\sqrt k}{E}.$$

The same holds for arrivals with $Y(t)=\mathbf 1\{N(t^-)<C\}$. Per bin $b$ and day type $d$, counts and exposures are summed over all days of that type. The naive estimator $k/T$ has expectation $\mu\,(1-\text{fraction of time empty})$, so it underestimates demand exactly when the station runs out. The confidence intervals are exact Poisson (Garwood) intervals:

$$\Big[\tfrac{1}{2E}\chi^2_{2k,\ \alpha/2},\ \ \tfrac{1}{2E}\chi^2_{2k+2,\ 1-\alpha/2}\Big].$$

### 1.3 Bin width (A2)

Bins are an approximation of a smoothly varying $\lambda(t)$. Narrow bins follow the curve but are noisy: relative error $\approx 1/\sqrt{k}$. Wide bins are stable but blur the peaks. Instead of guessing, fit on part of the training days and compute the **held-out log-likelihood** of the rest:

$$\ell_{\text{val}}=\sum_{j\in\text{val}}\log\hat r(t_j)\;-\;\int_{\text{val}}\hat r(t)\,Y(t)\,dt.$$

The width with the highest value predicts unseen days best.

In [ ]:
arr = arr[arr.N_before < C] 
cv = bd.select_bin_width(seg, arr, dep, C, train)
print(cv.round(4))
BIN = int(cv.loglik_total.idxmax())
print("chosen bin width:", BIN, "minutes")
rates = bd.estimate_rates(seg, arr, dep, C, BIN, days=train)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
truth = bd.true_binned_rates(BIN).loc["weekday"] if SYNTH else None
for ax, (col, name) in zip(axes, [("lam", "arrivals (births) λ"), ("mu", "departures (deaths) μ")]):
    r = rates.loc["weekday"]; h = r.index * BIN / 60
    ax.fill_between(h, r[f"{col}_lo"], r[f"{col}_hi"], step="post", color=C1, alpha=0.15, lw=0)
    ax.step(h, r[col], where="post", color=C1, label="corrected (exposure = available time)")
    ax.step(h, r[f"{col}_naive"], where="post", color=C2, lw=1.5, label="naive (exposure = all time)")
    if SYNTH:
        ax.step(h, truth[f"{col}_true"], where="post", color="0.25", lw=1, ls="--", label="true (synthetic)")
    ax.set_title(f"Weekday {name}"); ax.set_xlabel("hour of day"); ax.set_xticks(range(0, 25, 3))
axes[0].set_ylabel("events per hour"); axes[0].legend(fontsize=8, loc="upper right")
plt.show()
print(rates.loc["weekday"][["A", "D", "lam", "lam_naive", "mu", "mu_naive", "frac_empty", "frac_full"]].round(2).iloc[::max(1, 60 // BIN)])

### 1.4 Do days of the same type share a rate? (A5)

Under the model, the count on day $j$ in bin $b$ is $k_j\sim\text{Poisson}(r_b E_j)$, where $E_j$ is that day's available time in the bin. The Pearson statistic

$$X^2_b=\sum_{j}\frac{(k_j-\hat r_b E_j)^2}{\hat r_b E_j}\ \sim\ \chi^2_{J-1}\quad\text{under }H_0$$

compares the observed spread across days with the Poisson spread. The dispersion ratio $\varphi=X^2/\text{df}$ is $\approx1$ under $H_0$. With $\varphi>1$, days really differ (weather, events), and the counts follow a mixture. If the daily level is $\Lambda_d\sim\text{Gamma}(\alpha,\alpha)$, the counts are negative binomial with $\operatorname{Var}=\text{mean}+\text{mean}^2/\alpha$.

Day-bins where the station was available for less than half the bin are dropped. Their expected counts are tiny, which makes single Pearson terms explode even when the model is right. Results are pooled over bins by adding the $X^2$ and the df.

In [ ]:
res_between = {k: bd.between_day_dispersion(seg, ev, rates, k, C, days=train) for k, ev in (("arr", arr), ("dep", dep))}
for k, r in res_between.items():
    print(k, {a: round(float(b), 3) for a, b in bd.pooled(r).items()})

### 1.5 Is the process Poisson inside a bin? (A1, A2)

This test separates *within-day* behaviour from the day-level variation in 1.4. Condition on the total $K$ in one (day, bin). If events are Poisson with a constant rate inside the bin, then given $K$ they fall into the sub-slots $u$ **multinomially**, with probabilities proportional to available time:

$$(k_u)_u\,\big|\,K\ \sim\ \text{Multinomial}\Big(K,\ \tfrac{E_u}{\sum_v E_v}\Big),\qquad X^2=\sum_u\frac{(k_u-K\,w_u)^2}{K\,w_u}\ \sim\ \chi^2_{m-1}.$$

The daily level cancels out in the conditioning, so day-to-day variation can't trigger this test. Values $\varphi>1$ mean events bunch up inside the bin: either the rate changes within the bin (bins too wide, see 1.3) or the events are clustered (1.6).

In [ ]:
SUB = GRID_MIN if BIN > GRID_MIN else None
res_within = {}
if SUB:
    res_within = {k: bd.within_bin_dispersion(seg, ev, k, C, BIN, SUB, days=train) for k, ev in (("arr", arr), ("dep", dep))}
    for k, r in res_within.items():
        print(k, {a: round(float(b), 3) for a, b in bd.pooled(r).items()})
else:
    print("bin width equals the grid; rebuild segments with a finer GRID_MIN to run this test")

### 1.6 Do bikes move in groups? (A4)

Under a Poisson process with local rate $r$, the chance that an event follows the previous one within $w$ seconds is $1-e^{-rw}$. Adding over events gives the expected number of short gaps:

$$\mathbb E[\#\{\text{gaps}<w\}]=\sum_j\big(1-e^{-r(t_j)\,w}\big).$$

If the observed count is much higher (ratio > 1, small p-value), people arrive or leave together. The fix is a compound Poisson model, where each event moves a random number of bikes. That is still a Markov chain, with extra off-diagonal entries in $Q$.

In [ ]:
res_clu = {k: bd.clustering_check(ev[ev.date.isin(train)], rates, k, window_s=60) for k, ev in (("arr", arr), ("dep", dep))}
for k, r in res_clu.items():
    print(k, {a: (round(b, 3) if isinstance(b, float) else b) for a, b in r.items()})

### 1.7 Do rates depend on the state? (A3)

Group the states by bikes available (for departures) or free docks (for arrivals), e.g. $\{1\},\{2\},\{3,4\},\{5\ldots9\},\{10+\}$. Compare two multiplicative Poisson models for counts $k$ with exposure $E$ in each (cell, group, day):

$$H_0:\ r=a_{\text{cell}}\qquad\text{vs}\qquad H_1:\ r=a_{\text{cell}}\,b_{\text{group}},\qquad \text{LR}=2(\ell_1-\ell_0)\sim\chi^2_{G-1}.$$

Both are fitted exactly by iterative proportional fitting: each factor is rescaled until its fitted total equals its observed total, which is the Poisson MLE for a log-linear model. The table shows observed/expected events per group under $H_0$. A flat profile of $\approx1$ supports A3.

**Two traps.**
1. *Busy days.* Busy days also empty the station, so day-level variation looks like state dependence. A free effect per date does **not** fix this, because $n$ is driven by the same day's earlier events and fixed-effect Poisson is then biased. Instead, the exposures are scaled by an **outside** measure of the day's busyness, system-wide trips that day relative to the mean for that day type.
2. *Phantom bikes.* Reconstruction errors leave bikes in the count that aren't really there. That shows up as a drop **only** in the "1 bike" group (the station was truly empty). Genuine avoidance (riders reading the app) shows up as a gradual decline over 1, 2, 3–4. A broken last bike, which is common in practice, also looks like a drop at 1. So the robustness check repeats the test starting at 2 bikes.

In [ ]:
scale = bd.system_day_scale(df, HOLIDAYS)
res_state = {}
for k, ev in (("arr", arr), ("dep", dep)):
    res_state[k] = bd.state_dependence_test(seg, ev, k, C, BIN, days=train, day_scale=scale)
    rob = bd.state_dependence_test(seg, ev, k, C, BIN, days=train, edges=(2, 3, 5, 10), day_scale=scale)
    res_state[k]["p_from2"] = rob["p"]
    print(f"{k}: LR={res_state[k]['LR']:.1f}, df={res_state[k]['df']}, p={res_state[k]['p']:.4f}   "
          f"| starting at 2: p={rob['p']:.4f}")
    print(res_state[k]["table"].round(3), "\n")

### 1.8 Verdict

A rule of thumb for each check. A tiny p-value with an effect close to 1 (e.g. $\varphi=1.03$ on 10,000 events) is statistically significant but practically harmless. Judge by the size of the effect, not only by $p$.

In [ ]:
def verdict():
    rows = []
    for k in ("arr", "dep"):
        b = bd.pooled(res_between[k])
        rows.append((f"A5 days share rates ({k})", f"phi={b['phi']:.2f}, p={b['p']:.3g}",
                     "OK" if b["phi"] < 1.1 or b["p"] > 0.01 else "VIOLATED -> random daily level (Cox)"))
        if res_within:
            w = bd.pooled(res_within[k])
            rows.append((f"A1/A2 Poisson within bin ({k})", f"phi={w['phi']:.2f}, p={w['p']:.3g}",
                         "OK" if w["phi"] < 1.1 or w["p"] > 0.01 else "VIOLATED -> narrower bins or clustering"))
        c = res_clu[k]
        rows.append((f"A4 single moves ({k})", f"obs/exp={c['ratio']:.2f}, p={c['p_excess']:.3g}",
                     "OK" if c["ratio"] < 1.2 or c["p_excess"] > 0.01 else "VIOLATED -> batch (compound) arrivals"))
        s = res_state[k]; oe = s["table"]["obs/exp"]
        ok_all = s["p"] > 0.01 or (oe - 1).abs().max() < 0.1
        ok_from2 = s["p_from2"] > 0.01
        rows.append((f"A3 no state dependence ({k})",
                     f"p={s['p']:.3g} (from 2 bikes: {s['p_from2']:.3g}), obs/exp {oe.min():.2f}-{oe.max():.2f}",
                     "OK" if ok_all else ("OK from n>=2; drop at n=1 = phantom or broken last bike (1.7)" if ok_from2
                                          else "VIOLATED -> state-dependent rates")))
    return pd.DataFrame(rows, columns=["assumption", "evidence", "verdict"])
pd.set_option("display.max_colwidth", 80); verdict()

## Stage 2 — Compute the process

### 2.1 Transient distribution (Kolmogorov forward equations)

Let $p_n(t)=P(N(t)=n)$ as a row vector $\mathbf p(t)$. Balancing what enters and leaves state $n$ during $[t,t+h]$ and letting $h\to0$:

$$\frac{dp_n}{dt}=\lambda_{n-1}p_{n-1}-(\lambda_n+\mu_n)p_n+\mu_{n+1}p_{n+1}\quad\Longleftrightarrow\quad \frac{d\mathbf p}{dt}=\mathbf p(t)\,Q(t).$$

With $Q$ constant on each bin of length $\Delta$ hours, the solution is a product of matrix exponentials, **ordered** earliest first because matrices don't commute:

$$\mathbf p(t_0+m\Delta)=\mathbf p(t_0)\,e^{Q_{b_0}\Delta}\,e^{Q_{b_0+1}\Delta}\cdots e^{Q_{b_0+m-1}\Delta}.$$

The time unit of $Q$ and $\Delta$ must match. Rates are per hour here, so a 30-minute bin uses $\Delta=0.5$. Inside a bin we take several sub-steps so the curves are smooth. That changes only the plotting resolution, not the result.

The starting state is the median occupancy observed at 05:00 on training weekdays.

In [ ]:
DT = "weekday"
wk = train[bd.daytype_of(train, HOLIDAYS) == DT]
n0 = int(np.median(bd.n_at(occ, wk + pd.Timedelta(hours=5))))
ts, ps = bd.day_profile(rates, DT, C, n0, start_h=5, end_h=24, substeps=6)
cdf = ps.cumsum(1); mean = ps @ np.arange(C + 1)
q10, q90 = (cdf >= 0.1).argmax(1), (cdf >= 0.9).argmax(1)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(ts, ps[:, 0], color=C2, label="P(empty)")
axes[0].plot(ts, ps[:, C], color=C1, label="P(full)")
axes[0].set_ylim(0, 1); axes[0].set_title(f"{DT}, starting with {n0} bikes at 05:00"); axes[0].legend()
axes[1].fill_between(ts, q10, q90, color=C1, alpha=0.15, lw=0, label="10–90% range")
axes[1].plot(ts, mean, color=C1, label="E[N(t)]")
obs = np.array([bd.n_at(occ, wk + pd.Timedelta(hours=h)) for h in range(5, 24)])
axes[1].plot(range(5, 24), np.median(obs, 1), "o", color="0.3", ms=4, label="observed median (train)")
axes[1].axhline(C, color="0.5", lw=1, ls="--"); axes[1].set_title("bikes docked"); axes[1].legend(fontsize=8)
for ax in axes: ax.set_xlabel("hour of day"); ax.set_xticks(range(6, 25, 3))
plt.show()

### 2.2 Stationary law and whether it applies

If the rates stayed fixed, detailed balance across the cut between $n$ and $n+1$, $\pi_n\lambda=\pi_{n+1}\mu$, gives a truncated geometric law with $\rho=\lambda/\mu$:

$$\pi_n=\frac{(1-\rho)\rho^n}{1-\rho^{C+1}}\ (\rho\ne1),\qquad \pi_n=\tfrac{1}{C+1}\ (\rho=1).$$

That only describes a bin if the chain forgets its start within the bin. The eigenvalues of $-Q$ for this chain are $0$ and

$$\gamma_k=\lambda+\mu-2\sqrt{\lambda\mu}\cos\!\Big(\tfrac{k\pi}{C+1}\Big),\quad k=1,\dots,C,$$

so the **relaxation time** is $1/\gamma_1$. When it is longer than the bin, the stationary law is the wrong tool, and the transient computation of 2.1 is the one to use. The table below checks this for each bin.

In [ ]:
lam_b, mu_b = bd.bin_rates(rates, DT)
tab = pd.DataFrame({"hour": np.arange(len(lam_b)) * BIN / 60, "lam": lam_b, "mu": mu_b})
tab["rho"] = tab.lam / tab.mu.replace(0, np.nan)
pis = [bd.stationary(l, m, C) for l, m in zip(lam_b, mu_b)]
tab["pi_empty"] = [p[0] for p in pis]; tab["pi_full"] = [p[C] for p in pis]
tab["relax_h"] = 1 / np.array([bd.spectral_gap(l, m, C) for l, m in zip(lam_b, mu_b)])
tab["stationary_ok"] = tab.relax_h < BIN / 60
print(tab.iloc[::max(1, 60 // BIN)].round(3).to_string(index=False))

# check the closed forms against linear algebra for one busy bin
b = int(np.argmax(mu_b)); Q = bd.generator(lam_b[b], mu_b[b], C)
print("\ncheck at bin", b, ": |pi Q| =", np.abs(bd.stationary(lam_b[b], mu_b[b], C) @ Q).max(),
      "; gap closed form vs eig:", bd.spectral_gap(lam_b[b], mu_b[b], C), np.sort(np.linalg.eigvals(-Q).real)[1])

### 2.3 Expected time until the station empties or fills

Let $T_n$ be the expected time to reach $0$ from $n$, with constant rates. **First-step analysis**: the chain waits on average $1/(\lambda_n+\mu_n)$ and then jumps,

$$T_n=\frac{1}{\lambda_n+\mu_n}+\frac{\lambda_n}{\lambda_n+\mu_n}T_{n+1}+\frac{\mu_n}{\lambda_n+\mu_n}T_{n-1},\qquad T_0=0,\quad T_C=\tfrac1\mu+T_{C-1}.$$

With increments $u_n=T_n-T_{n-1}$, this becomes $u_C=1/\mu$ and $u_n=(1+\lambda u_{n+1})/\mu$, which unrolls to

$$u_n=\frac1\mu\sum_{k=0}^{C-n}\rho^k,\qquad T_n=\sum_{m=1}^n u_m.$$

The time to fill is the mirror image ($n\to C-n$, $\lambda\leftrightarrow\mu$). The cell computes time-to-empty at the pickup peak and time-to-full at the return peak, and checks them against a direct linear solve of $Q_{-0,-0}\,T=-\mathbf 1$.

In [ ]:
b_dep, b_arr = int(np.argmax(mu_b)), int(np.argmax(lam_b))
T_empty = bd.hitting_time_empty(lam_b[b_dep], mu_b[b_dep], C)
T_full = bd.hitting_time_full(lam_b[b_arr], mu_b[b_arr], C)
print(f"time to EMPTY uses the pickup peak ({b_dep * BIN / 60:.1f}h: lam={lam_b[b_dep]:.2f}, mu={mu_b[b_dep]:.2f} per hour)")
print(f"time to FULL  uses the return peak ({b_arr * BIN / 60:.1f}h: lam={lam_b[b_arr]:.2f}, mu={mu_b[b_arr]:.2f} per hour)")
print("max |closed form - linear solve|:",
      np.abs(T_empty - bd.hitting_time_linear(lam_b[b_dep], mu_b[b_dep], C, 0)).max(),
      np.abs(T_full - bd.hitting_time_linear(lam_b[b_arr], mu_b[b_arr], C, C)).max())
print(pd.DataFrame({"n": range(C + 1), "hours_to_empty": T_empty, "hours_to_full": T_full}).iloc[::2].round(2).to_string(index=False))
print("(these assume the peak rates last; read them as 'how fast it drains/fills at the peak')")

### 2.4 Best starting stock

Expected lost customers between $t_0$ and $t_1$, starting with $n_0$ bikes:

$$J(n_0)=\int_{t_0}^{t_1}\Big(\mu(t)\,p_0(t)+\lambda(t)\,p_C(t)\Big)dt,\qquad \mathbf p(t_0)=e_{n_0}^{\top}.$$

On a bin of length $\Delta$ with constant $Q$, the integral $\int_0^\Delta e^{Qs}ds$ comes exactly from one matrix exponential (Van Loan):

$$\exp\!\begin{pmatrix}Q\Delta & I\Delta\\ 0 & 0\end{pmatrix}=\begin{pmatrix}e^{Q\Delta} & \int_0^\Delta e^{Qs}ds\\ 0 & I\end{pmatrix}.$$

Using the whole transition matrix in place of one $\mathbf p$ gives $J(n_0)$ for **every** $n_0$ in a single pass:

$$\mathbf J=\sum_b \Phi_b\Big(\int_0^\Delta e^{Q_b s}ds\Big)\mathbf c_b,\qquad \Phi_b=\prod_{k<b}e^{Q_k\Delta},\quad \mathbf c_b=(\mu_b,0,\dots,0,\lambda_b)^\top.$$

This is the calculation behind a rebalancing target: the stock the trucks should leave before service starts.

In [ ]:
L = bd.losses_by_start_stock(rates, DT, C, start_h=5, end_h=24)
best = int(L.lost_total.idxmin())
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(L.index, L.lost_pickups, color=C2, label="lost pickups (station empty)")
ax.plot(L.index, L.lost_returns, color=C1, label="lost returns (station full)")
ax.plot(L.index, L.lost_total, color="0.2", label="total")
ax.axvline(best, color="0.5", lw=1, ls="--"); ax.text(best + 0.3, ax.get_ylim()[1] * 0.9, f"best n0 = {best}", fontsize=9)
ax.set_xlabel("bikes at 05:00"); ax.set_ylabel("expected lost customers, 05–24h"); ax.legend(fontsize=8)
plt.show()
print(L.round(2).iloc[::2])

### 2.5 Checking the computations

The matrix exponentials are verified by simulating the same chain directly (Gillespie): in state $n$, wait $\text{Exp}(\lambda_n+\mu_n)$, then move up with probability $\lambda_n/(\lambda_n+\mu_n)$. When a wait crosses a bin edge, restart the clock at the edge. That is exact because of memorylessness. The Monte Carlo frequencies should match the exact probabilities up to sampling noise ($\approx\sqrt{p(1-p)/R}$).

In [ ]:
rng = np.random.default_rng(0); R = 3000; grid = np.array([8, 9, 12, 18, 19, 21.0])
hits = np.zeros((len(grid), C + 1)); hours_empty = []
for _ in range(R):
    tt, nn = bd.simulate_ctmc(rates, DT, C, n0, 5, 24, rng)
    hits[np.arange(len(grid)), nn[np.searchsorted(tt, grid, side="right") - 1]] += 1
    hours_empty.append(np.diff(np.r_[tt, 24])[nn == 0].sum())
hits /= R
for g, h in zip(grid, hits):
    k = np.argmin(np.abs(ts - g))
    print(f"t={g:>4}: P(empty) exact {ps[k, 0]:.3f}  MC {h[0]:.3f} | P(full) exact {ps[k, C]:.3f}  MC {h[C]:.3f}")
print(f"hours empty from n0={n0}: exact {L.hours_empty[n0]:.3f}, MC {np.mean(hours_empty):.3f} ± {np.std(hours_empty) / np.sqrt(R):.3f}")

## Stage 3 — Evaluate on held-out days

The model was fitted on the first 80% of days. Everything here uses only the last 20%.

### 3.1 Time-rescaling test

Define the compensator $\Lambda(t)=\int_0^t \hat r(s)\,Y(s)\,ds$. The **time-rescaling theorem** says that if $\hat r(t)Y(t)$ is the true intensity, the rescaled gaps

$$z_j=\Lambda(t_j)-\Lambda(t_{j-1})\ \stackrel{\text{iid}}{\sim}\ \text{Exp}(1).$$

This checks the whole model at once: rates, bins, censoring and Poisson-ness. The diagnostics:

- **KS test** of $z$ against Exp(1). The p-value is valid because the rates come from other days.
- **QQ plot.** Too many small $z$ means clustering. A bend in the middle means the rate shape is wrong.
- **Lag-1 correlation of $z$**, about $0\pm 1/\sqrt n$ under $H_0$. A positive correlation means busy periods last longer than the model allows, which is the signature of day-level variation (A5).

In [ ]:
tr = {k: bd.time_rescaling(seg, ev, rates, k, C, test) for k, ev in (("arr", arr), ("dep", dep))}
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (k, r) in zip(axes, tr.items()):
    z = np.sort(r["z"]); q = -np.log(1 - (np.arange(1, len(z) + 1) - 0.5) / len(z))
    ax.plot(q, z, ".", ms=3, color=C1); m = max(q.max(), z.max()); ax.plot([0, m], [0, m], color="0.5", lw=1)
    ax.set_title(f"{k}: KS p={r['KS_p']:.3f}, lag-1 corr={r['lag1_corr']:.3f} (±{2 * r['lag1_se']:.3f})", fontsize=10)
    ax.set_xlabel("Exp(1) quantile"); ax.set_ylabel("rescaled gap")
plt.tight_layout(); plt.show()

### 3.2 Predictive accuracy

From the observed state $n_0=N(t_0)$ at each whole hour from 6 to 21 on test days, predict the distribution of $N(t_0+H)$ for $H=1,2,3$ hours:

$$\hat{\mathbf p}=e_{n_0}^{\top}\,\Phi(t_0,t_0+H).$$

Scores:
- **Log score** $-\log\hat p_{n_{\text{obs}}}$. Lower is better, and it is a proper scoring rule, so it can't be improved by hedging.
- **80% interval coverage.** Intervals of a discrete variable include their endpoints, so coverage above 80% is expected.
- **Mean absolute error** of $\mathbb E[N]$.

Two baselines show what the model adds:
- **naive rates**: same chain, but rates estimated without the censoring correction;
- **climatology**: the empirical distribution of $N$ at that clock time on training days, ignoring $n_0$.

In [ ]:
pred = bd.predictive_eval(occ, {"birth-death": rates, "birth-death naive": rates}, C, train, test, HOLIDAYS)
summary = pred.groupby(["H", "model"])[["logscore", "abs_err", "covered80"]].mean().unstack("model").round(3)
summary

### 3.3 Calibration

When the model says "30% chance of being empty in 2 hours", the station should be empty about 30% of those times. Predictions are grouped by predicted probability and compared with the observed frequency.

The **randomized PIT** $u=\hat F(n_{\text{obs}}-1)+V\,\hat p_{n_{\text{obs}}}$, with $V\sim U(0,1)$, is uniform when the predictive distribution is right. A U-shape means the predictions are too narrow (overconfident), a hump means too wide, and a slope means bias.

In [ ]:
m = pred[pred.model == "birth-death"]
ce, cf = bd.calibration_table(m, "p_empty", "empty"), bd.calibration_table(m, "p_full", "full", C)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for t_, name, col in ((ce, "empty", C2), (cf, "full", C1)):
    axes[0].plot(t_.mean_predicted, t_.observed_freq, "o-", color=col, label=f"P({name})")
axes[0].plot([0, 1], [0, 1], color="0.5", lw=1); axes[0].set_xlabel("predicted probability")
axes[0].set_ylabel("observed frequency"); axes[0].set_title("calibration"); axes[0].legend()
axes[1].hist(m.pit, bins=10, range=(0, 1), color=C1, edgecolor="white")
axes[1].axhline(len(m) / 10, color="0.5", lw=1, ls="--"); axes[1].set_title("randomized PIT (flat = calibrated)")
plt.tight_layout(); plt.show()
print(ce.round(3), "\n", cf.round(3))

## If a check fails

| Failed check | Meaning | Fix (keeps the birth–death machinery) |
|---|---|---|
| 1.4 between days, 3.1 positive lag-1 | days differ | Cox model: $\lambda_d(t)=\Lambda_d\,\lambda(t)$ with $\Lambda_d$ from system-wide trips or weather; or separate fits per weather class |
| 1.5 within bin | rate changes inside bins | narrower bins (1.3), or a smooth $\lambda(t)$ solved with an ODE solver |
| 1.6 clustering | group trips | compound Poisson: allow jumps of $\pm k$ in $Q$ with the observed group-size distribution |
| 1.7 state dependence | riders react to $n$ / broken bikes | use $\hat\lambda_n,\hat\mu_n$ per state group directly in $Q$ |
| 3.2 no better than climatology | model adds nothing beyond the clock | check reconstruction quality and rebalancing first, the usual culprits |

Real data will almost certainly show some day-to-day variation (1.4). The first extension worth adding is the Cox scaling, because it changes nothing else in the pipeline.